# SAKSHI — measured evaluation run (Kaggle, T4 + internet)
Attach the `sakshi-finetune-long` kernel as input (it holds `rfdetr_nano_{head,coco}.onnx`). Runs `scripts/gpu_runs.sh` on `main`: tests, head-model eval, door sweep, red-team video + sims, accuracy report, demo check. Result files land in `/kaggle/working/results` with their repo paths.

In [ ]:
import glob, json, os, subprocess
R = '/tmp/sakshi'
subprocess.run(f'rm -rf {R} && git clone -q --depth 1 https://github.com/nevil-sanish/SIH26245.git {R}', shell=True, check=True, cwd='/')
heads = glob.glob('/kaggle/input/**/rfdetr_nano_head.onnx', recursive=True)
assert heads, 'attach the sakshi-finetune-long kernel output'
print(heads[0]); print(subprocess.run('git -C /tmp/sakshi log --oneline -1', shell=True, capture_output=True, text=True).stdout)

## Full measured run (≈ 1.5–2 h)

In [ ]:
env = dict(os.environ, ONNX_DIR=os.path.dirname(heads[0]), RESULTS='/kaggle/working/results')
r = subprocess.run('bash scripts/gpu_runs.sh 2>&1 | grep -v -E "onnxruntime|EP Error|TensorRT" | tee /kaggle/working/gpu_runs.log | tail -60',
                   shell=True, cwd=R, env=env, executable='/bin/bash')
print('exit', r.returncode)

## Head model: tile + threshold chosen on valid, MAE on test

In [ ]:
h = json.load(open(f'{R}/data/work/head_onnx_check.json'))
print('deployed', h['deployed'])
for k, v in h['by_tile'].items(): print('tile', k, {x: v[x] for x in ('threshold', 'valid_mae', 'test_mae', 'test_bias')})

## Accuracy report headline (D4)

In [ ]:
rep = open(f'{R}/docs/ACCURACY_REPORT.md').read()
print(rep[rep.index('## Headline'):rep.index('## Per-scenario')])
print(rep[rep.index('## Failures'):rep.index('## Door events')])

## Demo check + bundled results

In [ ]:
log = open('/kaggle/working/gpu_runs.log').read()
print('DEMO_CHECK_OK' in log and 'demo check passed' or 'DEMO CHECK FAILED — see gpu_runs.log')
print('ALL_DONE' in log and 'pipeline complete' or 'pipeline did not finish')
print(subprocess.run('find /kaggle/working/results -type f | sort', shell=True, capture_output=True, text=True).stdout)